In [1]:
from PIL import Image
from keras.models import load_model
import numpy as np
from numpy import asarray
from numpy import expand_dims
import pickle
import cv2
import os.path
import numpy as np
from email.mime.text import MIMEText
from email.mime.image import MIMEImage
from email.mime.multipart import MIMEMultipart
import base64
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from keras_facenet import FaceNet

In [2]:
HaarCascade = cv2.CascadeClassifier(cv2.samples.findFile(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'))
MyFaceNet = FaceNet()

In [3]:
myfile = open("data.pkl", "rb")
database = pickle.load(myfile)
myfile.close()

In [4]:
# Email configuration
SENDER_EMAIL = '@gmail.com'
RECEIVER_EMAIL = '@gmail.com'

# get Gmail service with valid credentials
def get_gmail_service():
    creds = None
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json')
    
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file('credentials.json', ['https://www.googleapis.com/auth/gmail.send'])
            creds = flow.run_local_server(port=0)
        # Save the credentials for the next run
        with open('token.json', 'w') as token:
            token.write(creds.to_json())

    service = build('gmail', 'v1', credentials=creds)
    return service

# send email with attached image
def send_email_with_image(subject, body, image_path):
    service = get_gmail_service()

    
    message = MIMEMultipart()
    message['to'] = RECEIVER_EMAIL
    message['from'] = SENDER_EMAIL
    message['subject'] = subject

    # Attach text message
    message.attach(MIMEText(body))

    # Attach image
    with open(image_path, 'rb') as file:
        img = MIMEImage(file.read(), name=os.path.basename(image_path))
        message.attach(img)

    raw_message = base64.urlsafe_b64encode(message.as_bytes()).decode('utf-8')
    
    try:
        message = service.users().messages().send(userId='me', body={'raw': raw_message}).execute()
        print('Message Id: %s' % message['id'])
    except Exception as e:
        print('An error occurred: %s' % e)

In [6]:
cap = cv2.VideoCapture(0)
# variables for motion detection
prev_frame = None
motion_threshold = 1000
threshold = 1
identity = ' '  

# detect motion in the frame
def detect_motion(gbr1):
    global prev_frame
    gray = cv2.cvtColor(gbr1, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (21, 21), 0)
                                                               
    if 'prev_frame' not in detect_motion.__dict__:
        detect_motion.prev_frame = gray
        return False
    
    frame_diff = cv2.absdiff(detect_motion.prev_frame, gray)
    _, frame_diff = cv2.threshold(frame_diff, 30, 255, cv2.THRESH_BINARY)
    
    motion = np.sum(frame_diff) > 1000  
    detect_motion.prev_frame = gray                                            
    return motion


    # analyze texture of the face
def analyze_texture(gbr1):
    
    gray_face = cv2.cvtColor(gbr1, cv2.COLOR_BGR2GRAY)
    std_dev = np.std(gray_face)
    texture_threshold = 20  
    return std_dev > texture_threshold

# Loop over frames
while True:
    # Capture frame-by-frame
    ret, gbr1 = cap.read()
    
    # Detect face in the frame
    facey = HaarCascade.detectMultiScale(gbr1, 1.1, 4)
    
    if len(facey) > 0:
        # Extract first face found
        x1, y1, width, height = facey[0]
        
        x1, y1 = abs(x1), abs(y1)
        x2, y2 = x1 + width, y1 + height
        
        # Extract face region
        face = gbr1[y1:y2, x1:x2]
        
        # Convert face to RGB and PIL Image object
        face_pil = Image.fromarray(cv2.cvtColor(face, cv2.COLOR_BGR2RGB))
        
        # Resize face for FaceNet input size
        face_pil_resized = face_pil.resize((160, 160))
        
        # resized face back to numpy array
        face_resized = np.array(face_pil_resized)
        
        # liveness detection
        motion_detected = detect_motion(gbr1)
        texture_real = analyze_texture(gbr1)
        liveness_detected = motion_detected and texture_real
        
        # face recognition
        signature = MyFaceNet.embeddings(np.expand_dims(face_resized, axis=0))
        
         

        min_dist = 100
        #identity = ' '
        for key, value in database.items():
            dist = np.linalg.norm(value - signature)
            if dist < min_dist:                                     
                min_dist = dist
                identity = key
        

        # Check if the detected face is a known person or unknown
        if min_dist > threshold:
           identity = 'unknown'        
        
        # show recognized identity and bounding box
        cv2.putText(gbr1, identity, (100, 100), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 0), 2, cv2.LINE_AA)
        cv2.rectangle(gbr1, (x1, y1), (x2, y2), (0, 255, 0), 2)
        
        # show liveness detection result
        text = "Liveness: {}".format("Real" if liveness_detected else "Fake")
        cv2.putText(gbr1, text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    else:
        # show message if no face detected
        cv2.putText(gbr1, "No face detected", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
    
    # show the result frame             
    cv2.imshow('res', gbr1)
    

    # Capture and save image of unknown face
    if identity == 'unknown':
                                                  
        image_path = 'unknown_face.jpg'
        cv2.imwrite(image_path, gbr1)
            
        # Send email with the image saved 
        subject = 'Unknown Face Detected'
        body = 'An unknown face was detected by the system.'
        send_email_with_image(subject, body, image_path)
    
        
    cv2.imshow('res',gbr1)
    key = cv2.waitKey(1)
    if(key == ord('q')):
        break
       
cv2.destroyAllWindows()
cap.release() 

1/1 ━━━━━━━━━━━━━━━━━━━━ 8s 8s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━

RefreshError: ('invalid_grant: Token has been expired or revoked.', {'error': 'invalid_grant', 'error_description': 'Token has been expired or revoked.'})

: 

In [ ]:
#rak knty fl texture analyze 9ado lfundtion dylo oruni mn lwl